In [ ]:
import pandas as pd
import csv
import numpy as np
from pyhdf.SD import SD, SDC
import glob
import os
import time

In [ ]:
##lat, lon trans
rr = np.double(6371007.181)
xmin = -20015109. #rr * -np.pi
xmax =  20015109. #rr * np.pi
ymin = -10007554. #rr * -np.pi/2
ymax =  10007554. #rr * np.pi/2
tt = 1111950.     #(xmax - xmin) /36.
npixel = int(1200.)    # 250m 4800, 500m 2400, 1km 1200
res = tt / npixel
degree = 0.02

In [ ]:
def geomapping(hh,vv, npix):
    mlat= np.zeros((npix,npix))
    mlon= np.zeros((npix,npix))
    for jj in range(npix):
        yy = (9 - vv) * tt - (jj + 0.5) * res
        lat = yy * 180. / rr / np.pi
        mlat[jj,:] = lat
        for ii in range(npix):
            xx = (ii + 0.5) * res + hh * tt + xmin
            lon = (xx * 180. / np.pi) / (rr * np.cos(lat * np.pi / 180.))
            mlon[jj,ii] = lon
    return mlat, mlon

In [ ]:
def month_4mean(a,b,c,d):
    e = np.vstack((a,b,c,d))
    return np.nanmean(e)

def month_5mean(a,b,c,d,e):
    f = np.vstack((a,b,c,d,e))
    return np.nanmean(f)

# LST 

In [ ]:
tile = np.genfromtxt('../data/CRU/sn_bound_10deg.txt', skip_header = 7)
stnpath = '../data/CRU/station/'
MODpath = '../data/MODIS/'

#tile = np.genfromtxt('E:/DOCTOR/SB/ML/data/CRU/sn_bound_10deg.txt', skip_header = 7)
#stnpath = 'E:/DOCTOR/SB/ML/data/CRU/station/'
MOD_path = 'E:/DOCTOR/SB/ML/data/MODIS/'

dm_o = [1, 32, 60, 91, 121, 152, 182, 213, 244, 274, 305, 335]
dm_l = [1, 32, 61, 92, 122, 153, 183, 214, 245, 275, 306, 336]

dd_l = [[1, 9, 17, 25], [25, 33, 41, 49, 57], [57, 65, 73, 81, 89], [89, 97, 105, 113, 121], [121, 129, 137, 145], [153, 161, 169, 177], [177, 185, 193, 201, 209], [209, 217, 225, 233, 241], [241, 249, 257, 265, 273], [273, 281, 289, 297, 305], [305, 313, 321, 329], [329, 337, 345, 353, 361]]
dy_l = [[8, 8, 8, 7], [1, 8, 8, 8, 4], [4, 8, 8, 8, 3], [5, 8, 8, 8, 1], [7, 8, 8, 8], [8, 8, 8, 6], [2, 8, 8, 8, 5], [3, 8, 8, 8, 4], [4, 8, 8, 8, 2], [6, 8, 8, 8, 1], [7, 8, 8, 7], [1, 8, 8, 8, 6]] 

dd_o = [[1, 9, 17, 25], [25, 33, 41, 49, 57], [57, 65, 73, 81, 89], [89, 97, 105, 113], [121, 129, 137, 145], [145, 153, 161, 169, 177], [177, 185, 193, 201, 209], [209, 217, 225, 233, 241], [241, 249, 257, 265, 273], [273, 281, 289, 297], [305, 313, 321, 329], [329, 337, 345, 353, 361]]
dy_o = [[8, 8, 8, 7], [1, 8, 8, 8, 3], [5, 8, 8, 8, 2], [6, 8, 8, 8], [8, 8, 8, 7], [1, 8, 8, 8, 5], [3, 8, 8, 8, 4], [4, 8, 8, 8, 3], [5, 8, 8, 8, 1], [7, 8, 8, 8], [8, 8, 8, 6], [2, 8, 8, 8, 5]] 

stmm = 11
enmm = 11

for mm in np.arange(stmm,enmm+1):
    print(mm)
    
    ff = np.sort(glob.glob(stnpath + repr(mm) + '.csv'))
    ori = open(ff[0], 'r')
    rdr = csv.reader(ori)
    
    lines = {'STN':[], 'LAT':[], 'LON':[], 'YEAR':[], 'SAT':[], 'NDVI':[], 'LST_DAY':[], 'LST_NIGHT':[], 
             'NDVI_t':[], 'LST_DAY_t':[], 'LST_NIGHT_t':[], 'NDVI_a':[], 'LST_DAY_a':[], 'LST_NIGHT_a':[]}
    df = pd.DataFrame(lines)
    if not os.path.exists(stnpath + repr(mm) + '_data.csv'):
        df.to_csv(stnpath + repr(mm) + '_data.csv', mode='w',index=False)
    next(rdr)

## Read line
    for i,  line in enumerate(rdr):
        print(i)
        start = time.time()
        lat = float(line[1])
        lon = float(line[2])
        yy = int(line[3])
        #    print(lat,lon,yy)

## Find tile
        in_tile = False
        j = 0
        idx = (np.array([], dtype='int64'), np.array([], dtype='int64'))
        ii = idx[0]
        while(not any(ii)):
            while(not in_tile):
                in_tile = lat >= tile[j, 4] and lat <= tile[j, 5] and lon >= tile[j, 2] and lon <= tile[j, 3]
                j += 1
            vv = int(tile[j-1, 0])
            hh = int(tile[j-1, 1])
            #print('Vertical Tile:', vv, 'Horizontal Tile:', hh)

            mlat,mlon = geomapping(hh,vv,npixel)

            idx = np.where((mlat > lat - degree) * (mlat < lat + degree) * (mlon > lon - degree) * (mlon < lon + degree))
            ii = idx[0]
            in_tile = False
        
## Find station
        llon = mlon[idx]
        llat = mlat[idx]
        npoint = idx[0].size
        a,b=idx[0][:],idx[1][:]

        if yy % 4 == 0:
            dd = dd_l[mm-1]
            dy = dy_l[mm-1]
            dm = int(dm_l[mm-1])
        else:
            dd = dd_o[mm-1]
            dy = dy_o[mm-1]
            dm = int(dm_o[mm-1])
            
        nn = len(dd)
        
        result1 = np.zeros((nn,npoint))
        result2 = np.zeros((nn,npoint))
        result3 = np.zeros((nn,npoint))
        result4 = np.zeros((nn,npoint))
        
        ff = 0
        while( ff < nn):
#NDVI
#Terra
            flist = np.sort(glob.glob(MOD_path +'MOD13A3/' + repr(yy) + '/' +'MOD13A3.A' + repr(yy) + repr(dm).zfill(3) + '.h' + repr(hh).zfill(2) + 'v' + repr(vv).zfill(2) + '*.hdf'))
            nfiles = len(flist)
            res1 = line[5]
            if nfiles != 0:
                file = SD(flist[0], SDC.READ)
                datasets_dic = file.datasets()
                sds_obj = file.select('1 km monthly NDVI')
                ndvi_t = sds_obj.get()
                #jdx= np.where((ndvi>=10000) * (ndvi<=-2000))
                ndvi_t = ndvi_t / 10000.
                jdx = np.where((ndvi_t>=-0.3) * (ndvi_t<=0.))
                ndvi_t[jdx] = 0.
                res1 = ndvi_t[a[:],b[:]]
                #line[5] = np.nanmean(result)
                #print(line[5])
            
#Aqua
            flist = np.sort(glob.glob(MOD_path +'MYD13A3/' + repr(yy) + '/' +'MYD13A3.A' + repr(yy) + repr(dm).zfill(3) + '.h' + repr(hh).zfill(2) + 'v' + repr(vv).zfill(2) + '*.hdf'))
            nfiles = len(flist)
            res2 = line[5]
            if nfiles != 0:
                file = SD(flist[0], SDC.READ)
                datasets_dic = file.datasets()
                sds_obj = file.select('1 km monthly NDVI')
                ndvi_a = sds_obj.get()
                #jdx= np.where((ndvi>=10000) * (ndvi<=-2000))
                ndvi_a = ndvi_a / 10000.
                jdx = np.where((ndvi_a>=-0.3) * (ndvi_a<=0.))
                ndvi_a[jdx] = 0.
                res2 = ndvi_a[a[:],b[:]]
                #line[5] = np.nanmean(result)
                #print(line[5])

            
#LST           
#Terra
            flist = np.sort(glob.glob(MODpath +'MOD11A2/' + repr(yy) + '/' +'MOD11A2.A' + repr(yy) + repr(dd[ff]).zfill(3) + '.h' + repr(hh).zfill(2) + 'v' + repr(vv).zfill(2) + '*.hdf'))
            nfiles = len(flist)
            
            if nfiles != 0:
                file = SD(flist[0], SDC.READ)
                datasets_dic = file.datasets()
                
                sds_obj = file.select('LST_Day_1km')
                lst_day_t = sds_obj.get()
                lst_day_t = lst_day_t / 50.
                idx = np.where(lst_day_t <= 0.)
                lst_day_t[idx] = np.nan
                #print lst_day[a[:],b[:]]
                result1[ff,:] = lst_day_t[a[:],b[:]]

                sds_obj = file.select('LST_Night_1km')
                lst_night_t = sds_obj.get()
                lst_night_t = lst_night_t / 50.
                idx = np.where(lst_night_t <= 0.)
                lst_night_t[idx] = np.nan
                #print lst_night[a[:],b[:]]
                result2[ff,:] = lst_night_t[a[:],b[:]]
#Aqua
            flist = np.sort(glob.glob(MODpath +'MYD11A2/'  + repr(yy) + '/' +'MYD11A2.A' + repr(yy) + repr(dd[ff]).zfill(3) + '.h' + repr(hh).zfill(2) + 'v' + repr(vv).zfill(2) + '*.hdf'))
            nfiles = len(flist)
            
            if nfiles != 0:
                file = SD(flist[0], SDC.READ)
                datasets_dic = file.datasets()
                
                sds_obj = file.select('LST_Day_1km')
                lst_day_a = sds_obj.get()
                lst_day_a = lst_day_a / 50.
                idx = np.where(lst_day_a <= 0.)
                lst_day_a[idx] = np.nan
                #print lst_day[a[:],b[:]]
                result3[ff,:] = lst_day_a[a[:],b[:]]

                sds_obj = file.select('LST_Night_1km')
                lst_night_a = sds_obj.get()
                lst_night_a = lst_night_a / 50.
                idx = np.where(lst_night_a <= 0.)
                lst_night_a[idx] = np.nan
                #print lst_night[a[:],b[:]]
                result4[ff,:] = lst_night_a[a[:],b[:]]
                                
            ff += 1

#make monthly
        if nn == 4:
            a = np.tile(result1[0,:],(dy[0],1))
            b = np.tile(result1[1,:],(dy[1],1))
            c = np.tile(result1[2,:],(dy[2],1))
            d = np.tile(result1[3,:],(dy[3],1))
            day_t = month_4mean(a,b,c,d) - 273.15
            if day_t == -273.15:
                day_t = 0
            a = np.tile(result2[0,:],(dy[0],1))
            b = np.tile(result2[1,:],(dy[1],1))
            c = np.tile(result2[2,:],(dy[2],1))
            d = np.tile(result2[3,:],(dy[3],1))
            night_t = month_4mean(a,b,c,d) - 273.15
            if night_t == -273.15:
                night_t = 0
            a = np.tile(result3[0,:],(dy[0],1))
            b = np.tile(result3[1,:],(dy[1],1))
            c = np.tile(result3[2,:],(dy[2],1))
            d = np.tile(result3[3,:],(dy[3],1))
            day_a = month_4mean(a,b,c,d) - 273.15
            if day_a == -273.15:
                day_a = 0
            a = np.tile(result4[0,:],(dy[0],1))
            b = np.tile(result4[1,:],(dy[1],1))
            c = np.tile(result4[2,:],(dy[2],1))
            d = np.tile(result4[3,:],(dy[3],1))
            night_a = month_4mean(a,b,c,d) - 273.15
            if night_a == -273.15:
                night_a = 0
            
        if nn ==  5:
            a = np.tile(result1[0,:],(dy[0],1))
            b = np.tile(result1[1,:],(dy[1],1))
            c = np.tile(result1[2,:],(dy[2],1))
            d = np.tile(result1[3,:],(dy[3],1))
            e = np.tile(result1[4,:],(dy[4],1))
            day_t = month_5mean(a,b,c,d,e) - 273.15
            if day_t == -273.15:
                day_t = 0
            a = np.tile(result2[0,:],(dy[0],1))
            b = np.tile(result2[1,:],(dy[1],1))
            c = np.tile(result2[2,:],(dy[2],1))
            d = np.tile(result2[3,:],(dy[3],1))
            e = np.tile(result2[4,:],(dy[4],1))
            night_t = month_5mean(a,b,c,d,e) - 273.15          
            if night_t == -273.15:
                night_t = 0
            a = np.tile(result3[0,:],(dy[0],1))
            b = np.tile(result3[1,:],(dy[1],1))
            c = np.tile(result3[2,:],(dy[2],1))
            d = np.tile(result3[3,:],(dy[3],1))
            e = np.tile(result3[4,:],(dy[4],1))
            day_a = month_5mean(a,b,c,d,e) - 273.15
            if day_a == -273.15:
                day_a = 0
            a = np.tile(result4[0,:],(dy[0],1))
            b = np.tile(result4[1,:],(dy[1],1))
            c = np.tile(result4[2,:],(dy[2],1))
            d = np.tile(result4[3,:],(dy[3],1))
            e = np.tile(result4[4,:],(dy[4],1))
            night_a = month_5mean(a,b,c,d,e) - 273.15          
            if night_a == -273.15:
                night_a = 0
        if res1 =='':
            res1 = np.nan
        if res2 =='':
            res2 = np.nan
        lines = {'STN':[line[0]], 'LAT':[line[1]], 'LON':[line[2]], 'YEAR':[line[3]], 'SAT':[line[4]],
                 'NDVI':[line[5]], 'LST_DAY':[line[6]], 'LST_NIGHT':[line[7]],
                 'NDVI_t':[np.nanmean(res1)], 'LST_DAY_t':[day_t], 'LST_NIGHT_t':[night_t],
                 'NDVI_a':[np.nanmean(res2)], 'LST_DAY_a':[day_a], 'LST_NIGHT_a':[night_a]}

        df=pd.DataFrame(lines)
        df = df.fillna(0)
        df.to_csv(stnpath + repr(mm) + '_data.csv', mode='a',index=False, header=False)
        print("time :", time.time() - start)
#    wr.writerows(lines)
#    f.close()